# Fink/LSST — Reload and Display Saved Light Curves (Tags × DDFs)

This notebook reads the Parquet files saved by `07_fink_tags_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per Fink tag.

**Expected data** in `data_FINK_BLOCK_LC_07/`:
- `{tag}_fp.parquet`  — forced-photometry light curves
- `{tag}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics
- `tag_ranking.csv`   — variability ranking by tag
- `visit_summary_src.csv` / `visit_summary_fp.csv` — visit-level summaries

No API call is made in this notebook.

## Differences from notebook 02 (block replot)

| Aspect | NB 02 (blocks) | NB 08 (tags) |
|--------|---------------|-------------|
| Source data dir | `data_FINK_BLOCK_LC_01` | `data_FINK_BLOCK_LC_07` |
| `lc_cache` structure | `lc_cache[group][oid]` | `lc_cache[oid]` with `group`/`tag`/`field` keys |
| Group key | `classify_object()` category | Fink tag name directly |
| `field` column | yes | yes (DDF name, identical semantics) |
| `tag` column | absent | present (= group) |
| Flatness filtering | `CALIBRATION_EXCLUDE` set | all tags kept (no calibration filter) |
| Extra plots | — | per-DDF flatness boxplot, tag ranking table |


## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import collections
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07"
DIR_DATA = f"data_{NB_TAG}"  # input:  written by notebook 07
DIR_FIGS = f"figs_{NB_TAG}_08"  # output: figures specific to this replot notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per tag
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ─────────────────────────────────────────────────────────────
# Choose which tags to display:
#   'all'  -> all tags found on disk (default)
#   list   -> subset of tags, e.g. ['in_tns', 'hostless_candidate']
PLOT_MODE = "all"  # <── change here: 'all' | list of tags

# ── Known Fink tag descriptions (used for display) ───────────────────────────
FINK_TAGS = {
    "extragalactic_lt20mag_candidate": "Rising, bright (mag < 20), extragalactic candidates",
    "extragalactic_new_candidate": "New (< 48 h first detection) and potentially extragalactic",
    "hostless_candidate": "Hostless alerts according to ELEPHANT (arXiv:2404.18165)",
    "in_tns": "Alerts with a known counterpart in TNS (AT or confirmed)",
    "sn_near_galaxy_candidate": "Alerts matching a galaxy catalog and consistent with SNe",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions (+ luptitude support)

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties."""
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

In [ ]:
def flux_to_luptitude(flux_nJy, flux_err_nJy, zero_point=31.4):
    """Convert PSF flux (nJy) to asinh magnitudes (Luptitudes).

    Luptitudes (Lupton et al. 1999) handle zero and negative flux values
    that arise in difference-image photometry (DIA).  They behave as
    standard AB magnitudes at high signal-to-noise and transition to a
    linear flux scale near the noise floor, so every measurement is
    plotted -- including those with negative flux.

    The softening parameter *b* is set to the median flux uncertainty of
    the input array, placing the linear/log transition at the 1-sigma
    noise level.  This matches the convention used in the DP2 DiaObject
    Butler notebook (plot_dia_object_luptitudes).

    Formula
    -------
    mu = -2.5 * log10(e) * [arcsinh(f / 2b) + ln(b)] + (zp + 2.5*log10(b))

    Error propagation
    -----------------
    sigma_mu = 1.085736 * sigma_f / sqrt(f^2 + (2b)^2)

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.  May contain negative values.
    flux_err_nJy : array-like
        1-sigma flux uncertainty in nJy (used to derive the softening
        parameter b = median(flux_err_nJy)).
    zero_point : float, optional
        AB zero-point offset in nJy units (default 31.4 for nJy).

    Returns
    -------
    lup : ndarray
        Asinh magnitude (Luptitude) for each measurement.
    lup_err : ndarray
        1-sigma uncertainty on the Luptitude.
    b : float
        Softening parameter used (= median of flux_err_nJy).
    """
    flux = np.asarray(flux_nJy, dtype=float)
    err = np.asarray(flux_err_nJy, dtype=float)

    # Softening parameter: median noise level of the input data
    b = float(np.nanmedian(err))
    if not np.isfinite(b) or b <= 0.0:
        b = 1.0  # safe fallback

    # Pre-factor  2.5 / ln(10) ~ 1.085736
    log10_e_inv = 1.085736

    # Asinh magnitude (Luptitude)
    lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2.0 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))

    # Error propagation
    lup_err = log10_e_inv * err / np.sqrt(flux**2 + (2.0 * b) ** 2)

    return lup, lup_err, b


print("flux_to_luptitude() defined.")

## 3. Discover available tags from Parquet files on disk

In [ ]:
# Auto-discover available tags from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def tag_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


tags_fp = {tag_from_path(p, "_fp.parquet"): p for p in fp_files}
tags_src = {tag_from_path(p, "_src.parquet"): p for p in src_files}
all_tags = sorted(set(tags_fp) | set(tags_src))

# Apply PLOT_MODE filter
if PLOT_MODE == "all":
    selected_tags = list(all_tags)
elif isinstance(PLOT_MODE, list):
    selected_tags = [t for t in all_tags if t in PLOT_MODE]
else:
    selected_tags = list(all_tags)
    print(f"WARNING: unknown PLOT_MODE={PLOT_MODE!r}, defaulting to 'all'")

print(f"Tags on disk: {len(all_tags)},  selected (PLOT_MODE={PLOT_MODE!r}): {len(selected_tags)}")
print()
for t in all_tags:
    has_fp = "yes" if t in tags_fp else "no "
    has_src = "yes" if t in tags_src else "no "
    sel = "<<" if t in selected_tags else "  "
    desc = FINK_TAGS.get(t, "(unknown tag)")
    print(f"  {sel} {t:50s}  fp={has_fp}  src={has_src}  |  {desc}")

## 4. Load Parquet files — reconstruct `lc_cache`

The cache structure mirrors notebook 07:
```python
lc_cache[oid] = {
    'fp'   : DataFrame,   # forced photometry
    'src'  : DataFrame,   # diaSources
    'group': str,         # = tag name
    'tag'  : str,         # = tag name
    'field': str,         # DDF name
}
```

In [ ]:
lc_cache = {}  # oid (str) → {fp, src, group, tag, field}

for tag in all_tags:
    for lc_type, path_dict in (("fp", tags_fp), ("src", tags_src)):
        if tag not in path_dict:
            continue
        df = pd.read_parquet(path_dict[tag])

        # Drop residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag/mag_err if absent
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        # The Parquet files written by NB07 contain 'diaObjectId', 'group', 'tag', 'field' columns.
        oid_col = "diaObjectId" if "diaObjectId" in df.columns else "r:diaObjectId"

        for oid, sub in df.groupby(oid_col):
            oid = str(oid)
            if oid not in lc_cache:
                # Retrieve metadata from the first row (same for all rows of an oid)
                row0 = sub.iloc[0]
                lc_cache[oid] = {
                    "fp": pd.DataFrame(),
                    "src": pd.DataFrame(),
                    "group": str(row0.get("group", tag)),
                    "tag": str(row0.get("tag", tag)),
                    "field": str(row0.get("field", "unknown")),
                }
            lc_cache[oid][lc_type] = sub.reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total objects loaded: {len(lc_cache)}")
print()

# Build group_oids index: tag -> list of oids
all_groups = sorted(set(d["group"] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d["group"]].append(oid)

print("Objects loaded per tag   [DDF breakdown]:")
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    fields_in_group = collections.Counter(lc_cache[o]["field"] for o in group_oids[g])
    field_str = "  ".join(f"{f}:{n}" for f, n in sorted(fields_in_group.items()))
    n_pts = sum(len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"]) for o in group_oids[g])
    print(f"  {g:50s}  {len(group_oids[g]):3d} objects  {n_pts:6d} pts   {field_str}")

## 5. Load flatness metrics and tag ranking

In [ ]:
# ── Flatness metrics ──────────────────────────────────────────────────────────
csv_path = os.path.join(DIR_DATA, "flatness_metrics.csv")
if os.path.exists(csv_path):
    df_flat = pd.read_csv(csv_path)
    print(f"flatness_metrics.csv loaded: {len(df_flat)} rows")
    print("\nMedian flatness by tag:")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
    print("\nMedian flatness by DDF:")
    if "field" in df_flat.columns:
        print(df_flat.groupby("field")[["n_pts", "rms_var"]].median().round(4))
else:
    df_flat = pd.DataFrame()
    print("flatness_metrics.csv not found — flatness plots will be skipped.")

# ── Tag ranking ───────────────────────────────────────────────────────────────
ranking_path = os.path.join(DIR_DATA, "tag_ranking.csv")
if os.path.exists(ranking_path):
    df_ranking = pd.read_csv(ranking_path)
    print(f"\ntag_ranking.csv loaded: {len(df_ranking)} rows")
else:
    df_ranking = pd.DataFrame()
    print("\ntag_ranking.csv not found.")

## 6. Flatness boxplot per tag

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle(
        "Flux variability by Fink tag (all DDFs combined) — lower = flatter",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_tag")
    plt.show()

## 6b. Flatness boxplot per DDF (all tags combined)

In [ ]:
if df_flat.empty or "field" not in df_flat.columns:
    print("No flatness / field data available.")
else:
    fields_present = sorted(df_flat["field"].dropna().unique())
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_field = [df_b[df_b["field"] == f]["rms_var"].dropna().values for f in fields_present]
        bp = ax.boxplot(
            data_per_field, labels=fields_present, patch_artist=True, notch=False, showfliers=True
        )
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=8)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by DDF (all tags combined)", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01b_flatness_boxplot_by_field")
    plt.show()

## 7. Light-curve plotting functions

In [ ]:
def rank_oids(oid_list, nc=NC):
    """Return the top-nc object IDs, ranked by total number of data points."""
    scored = [(o, len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"])) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def getTminTmax(df, df_src, df_fp):
    """ """

    # find tmin, tmax on all points / all bands
    t = df["r:midpointMjdTai"].values
    t_src = df_src["r:midpointMjdTai"].values
    t_fp = df_fp["r:midpointMjdTai"].values

    # find the first point
    if len(t_src) > 0:
        tmin = t_src.min()
    else:
        tmin = t.min()
    tmax = t.max()
    return tmin, tmax


def plot_lc_grid(oid_list, group, mode="flux", nc=NC):
    """
    Plot a (n_objects x n_bands) grid of light curves for one Fink tag.

    Parameters
    ----------
    oid_list : list of str
        Object IDs belonging to this tag.
    group : str
        Tag name (used for title and file name).
    mode : str
        Display mode:
        - 'flux'      : raw PSF flux in nJy.
        - 'mag'       : AB magnitude (NaN for non-positive flux).
        - 'luptitude' : asinh magnitude, handles zero and negative
                        differential flux gracefully (recommended for DIA).
    nc : int
        Maximum number of objects to plot.

    Notes on 'luptitude' mode
    -------------------------
    Luptitudes (Lupton et al. 1999) replace the logarithm with an inverse
    hyperbolic sine so that the scale is logarithmic (magnitude-like) at
    high flux and linear near zero.  Every measurement is plotted,
    including those with negative flux arising from DIA noise fluctuations.
    The softening parameter b is set per-band to the median flux
    uncertainty, following the convention of the DP2 DiaObject Butler
    notebook (plot_dia_object_luptitudes).
    """

    STYLE = {
        "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),  # ▽ triangle inversé vide
        "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),  # ● disque plein
    }

    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for tag {group}.")
        return

    fig, axes = plt.subplots(
        n_obj,
        len(BANDS) + 1,
        figsize=(2.8 * (len(BANDS) + 1), 2.6 * n_obj),
        sharex=False,
        sharey=False,
        squeeze=False,
    )

    def clean(df):
        if len(df) > 0:
            return (
                df.drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
                .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
                .sort_values("r:midpointMjdTai")
                .reset_index(drop=True)
            )
        else:
            return df

    # loop on objects
    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        field = d.get("field", "")

        frames = [df for df in (d["fp"], d["src"]) if not df.empty]
        if not frames:
            continue

        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        # df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
        #    drop=True
        # )
        df_all = clean(df_all)
        df_src = clean(d["src"])
        df_fp = clean(d["fp"])

        # find tmin, tmax on all points / all bands
        t = df_all["r:midpointMjdTai"].values
        t_src = df_src["r:midpointMjdTai"].values
        t_fp = df_fp["r:midpointMjdTai"].values

        # find the first point
        # if len(t_src) > 0:
        #    tmin = t_src.min()
        # else:
        #    tmin = t.min()
        # tmax = t.max()
        tmin, tmax = getTminTmax(df_all, df_src, df_fp)

        dtmin = t.min() - tmin
        dtmax = tmax - tmin

        # Loop on bands of the objects

        ax_first_band = None
        ax_last_allbands = axes[row_idx][len(BANDS)]
        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")
            df_b_src = df_src[df_src["r:band"] == band].sort_values("r:midpointMjdTai")
            df_b_fp = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")

            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            if ax_first_band is None:
                ax_first_band = ax

            # Select y-values according to the requested display mode
            if mode == "flux":
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
                mask_src = np.isfinite(df_b_src["r:psfFlux"].values) & np.isfinite(
                    df_b_src["r:psfFluxErr"].values
                )
                df_b_src = df_b_src[mask_src].reset_index(drop=True)
                mask_fp = np.isfinite(df_b_fp["r:psfFlux"].values) & np.isfinite(
                    df_b_fp["r:psfFluxErr"].values
                )
                df_b = df_b[mask].reset_index(drop=True)

                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue

                # extract flux yvalues to plot
                y = df_b["r:psfFlux"].values
                yerr = df_b["r:psfFluxErr"].values

                y_src = df_b_src["r:psfFlux"].values
                yerr_src = df_b_src["r:psfFluxErr"].values

                y_fp = df_b_fp["r:psfFlux"].values
                yerr_fp = df_b_fp["r:psfFluxErr"].values

            elif mode == "mag":
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
                df_b = df_b[mask].reset_index(drop=True)

                mask_src = np.isfinite(df_b_src["mag"].values) & np.isfinite(df_b_src["mag_err"].values)
                df_b_src = df_b_src[mask_src].reset_index(drop=True)

                mask_fp = np.isfinite(df_b_fp["mag"].values) & np.isfinite(df_b_fp["mag_err"].values)
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True)

                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue

                # extract mag yvalues to plot
                y = df_b["mag"].values
                yerr = df_b["mag_err"].values

                y_src = df_b_src["mag"].values
                yerr_src = df_b_src["mag_err"].values

                y_fp = df_b_fp["mag"].values
                yerr_fp = df_b_fp["mag_err"].values

                # invert the y axis for mag
                ax.invert_yaxis()

            elif mode == "luptitude":
                # Asinh magnitude: handles zero and negative flux values.
                # All finite measurements are kept (no flux > 0 requirement).
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
                df_b = df_b[mask].reset_index(drop=True)

                mask_src = np.isfinite(df_b_src["r:psfFlux"].values) & np.isfinite(
                    df_b_src["r:psfFluxErr"].values
                )
                df_b_src = df_b_src[mask_src].reset_index(drop=True)

                mask_fp = np.isfinite(df_b_fp["r:psfFlux"].values) & np.isfinite(
                    df_b_fp["r:psfFluxErr"].values
                )
                df_b_fp = df_b_fp[mask_fp].reset_index(drop=True)

                if len(df_b) < 3:
                    ax.set_visible(False)
                    continue

                # extract luptitude y values to plot
                y, yerr, b_soft = flux_to_luptitude(df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values)

                y_src, yerr_src, b_soft_src = flux_to_luptitude(
                    df_b_src["r:psfFlux"].values, df_b_src["r:psfFluxErr"].values
                )
                y_fp, yerr_fp, b_soft_fp = flux_to_luptitude(
                    df_b_fp["r:psfFlux"].values, df_b_fp["r:psfFluxErr"].values
                )

                # invert the y axis for luptitudes
                ax.invert_yaxis()  # brighter = lower numeric value, same as magnitudes

            else:
                raise ValueError(f"Unknown mode {mode!r}. Choose 'flux', 'mag', or 'luptitude'.")

            # extract time-xvalues to plot
            t = df_b["r:midpointMjdTai"].values
            t_src = df_b_src["r:midpointMjdTai"].values
            t_fp = df_b_fp["r:midpointMjdTai"].values

            dt = t - tmin
            dt_src = t_src - tmin
            dt_fp = t_fp - tmin

            color = BAND_COLORS.get(band, "gray")

            # Do the plot here with errorbar
            # ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)
            # plot the current band
            ax.errorbar(dt_src, y_src, yerr=yerr_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"])
            ax.errorbar(dt_fp, y_fp, yerr=yerr_fp, lw=0, elinewidth=0.5, color=color, **STYLE["fp"])

            # plot the rightmost subplot will all band curves shown
            ax_last_allbands.errorbar(
                dt_src, y_src, yerr=yerr_src, lw=1.0, elinewidth=1.2, color=color, **STYLE["src"]
            )
            ax_last_allbands.errorbar(
                dt_fp, y_fp, yerr=yerr_fp, lw=0, elinewidth=0.5, color=color, **STYLE["fp"]
            )

            # calculate the variability
            rms = rms_variability(df_b["r:psfFlux"].values)

            # Sub-title: show softening parameter for luptitudes
            if mode == "luptitude":
                ax.set_title(f"{band} N={len(df_b)} b={b_soft:.1f}nJy", fontsize=7, pad=2, color=color)
            else:
                ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}", fontsize=7, pad=2, color=color)

            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)
            ax.set_xlim(dtmin - 2, dtmax + 2)

        # Y-axis label: object ID + DDF field + display unit
        _ylabel_map = {
            "flux": "flux (nJy)",
            "mag": "AB mag",
            "luptitude": "asinh mag (luptitude)",
        }
        ylabel_unit = _ylabel_map.get(mode, mode)

        label_ax = ax_first_band if ax_first_band is not None else axes[row_idx][0]
        field_label = f"  [{field}]" if field else ""
        # label_ax.set_ylabel(
        #    f"{oid}{field_label}\n{ylabel_unit}",
        #    fontsize=9,
        # )

        if ax_first_band is not None:
            deep_field = d["fp"]["field"].unique().item() if not d["fp"].empty else ""
            field_label = f"  [{deep_field}]" if deep_field else ""
            ax_first_band.set_ylabel(f"{oid}{field_label}\n{ylabel_unit}", fontsize=10)

        else:
            axes[row_idx][0].set_ylabel(f"{oid}\n{ylabel_unit}", fontsize=8)

        deep_field = d["fp"]["field"].unique().item() if not d["fp"].empty else ""
        ax_last_allbands.set_xlabel("Δt (days)", fontsize=7)
        ax_last_allbands.tick_params(labelsize=7)
        ax_last_allbands.set_ylabel(f"{ylabel_unit}", fontsize=10)
        ax_last_allbands.set_title(f"{oid} ({deep_field})", fontsize=8)
        ax_last_allbands.set_xlim(dtmin - 2, dtmax + 2)

    fig.suptitle(f"Tag: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plotting functions defined (modes: flux / mag / luptitude).")

## 8. Light curves in flux (nJy)

Controlled by `PLOT_MODE` (section 1): `'all'` · list of tags.

In [ ]:
groups_to_plot = [g for g in selected_tags if g in group_oids and len(group_oids[g]) >= 1]

for group in groups_to_plot:
    n_obj = len(group_oids[group])
    desc = FINK_TAGS.get(group, "")
    print(f"\n=== {group} ({n_obj} objects) — flux ===")
    if desc:
        print(f"    {desc}")
    plot_lc_grid(group_oids[group], group, mode="flux")

## 9. Light curves in AB magnitude

Same `PLOT_MODE` selection as section 8.  Measurements with non-positive flux are
dropped (NaN).  Use section 9b (luptitudes) to visualise all data points.

In [ ]:
for group in groups_to_plot:
    n_obj = len(group_oids[group])
    print(f"\n=== {group} ({n_obj} objects) — magnitude ===")
    plot_lc_grid(group_oids[group], group, mode="mag")

## 9b. Light curves as Luptitudes (asinh magnitudes)

Luptitudes handle zero and negative PSF flux values that routinely appear
in difference-image photometry (DIA).  The conversion follows Lupton et al.
(1999) with a per-band softening parameter *b* equal to the median flux
uncertainty.  Every measurement is plotted -- there are no missing points
due to non-positive flux -- making it easier to judge whether variability
is real or just noise fluctuation around zero.

Same `PLOT_MODE` selection as sections 8--9.

In [ ]:
# Luptitude light curves -- controlled by PLOT_MODE (section 1)
# Every data point is plotted (no NaN due to non-positive flux).
for group in groups_to_plot:
    n_obj = len(group_oids[group])
    desc = FINK_TAGS.get(group, "")
    print(f"\n=== {group} ({n_obj} objects) — luptitude ===")
    if desc:
        print(f"    {desc}")
    plot_lc_grid(group_oids[group], group, mode="luptitude")

## 10. Detailed view — single object inspector (flux + magnitude + luptitude)

Set `TARGET_TAG` and `TARGET_OID` below to inspect any individual object.
The detailed plot now shows **three panels** per band:

1. **Flux (nJy)** — raw PSF flux with zero-flux reference line.
2. **AB magnitude** — standard log scale (NaN for non-positive flux).
3. **Luptitude** — asinh magnitude that includes all measurements.


In [ ]:
def plot_singleobjlightcurve(
    lc_cache,
    target_group,
    target_oid,
    bands,
    band_colors,
    modes=("flux", "mag", "luptitude"),  # panels à afficher
    show_per_band=True,
    show_combined=True,
    style=None,
    save=False,
    filename_prefix="03_detail",
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if style is None:
        style = {
            "fp": dict(fmt="v", ms=4, mfc="none", alpha=0.75, label="fp"),
            "src": dict(fmt="o", ms=3, mfc=None, alpha=0.85, label="src"),
        }

    d = lc_cache[target_group][target_oid]

    # ── Data prep ─────────────────────────────────────────────
    frames = [df for df in (d["fp"], d["src"]) if not df.empty]

    df_obj = (
        pd.concat(frames, ignore_index=True)
        .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
        .sort_values("r:midpointMjdTai")
        .reset_index(drop=True)
    )

    def clean(df):
        return (
            df.drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
            .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
            .sort_values("r:midpointMjdTai")
            .reset_index(drop=True)
        )

    df_obj_src = clean(d["src"])
    df_obj_fp = clean(d["fp"])

    tmin, tmax = getTminTmax(df_obj, df_obj_src, df_obj_fp)

    bands_obj = [b for b in bands if b in df_obj["r:band"].values]

    # ── Layout logic ─────────────────────────────────────────
    n_rows = 0
    if show_per_band:
        n_rows += len(bands_obj)
    if show_combined:
        n_rows += 1

    n_cols = len(modes)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.2 * n_rows), squeeze=False)

    # mapping mode → colonne
    mode_to_col = {m: i for i, m in enumerate(modes)}

    current_row = 0

    # ── Core plotting function ───────────────────────────────
    def plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color, is_combined=False):
        t = df_b["r:midpointMjdTai"].values
        dt = t - tmin

        t_src = df_b_src["r:midpointMjdTai"].values
        dt_src = t_src - tmin

        t_fp = df_b_fp["r:midpointMjdTai"].values
        dt_fp = t_fp - tmin

        if mode == "flux":
            mask_src = np.isfinite(df_b_src["r:psfFlux"]) & np.isfinite(df_b_src["r:psfFluxErr"])
            mask_fp = np.isfinite(df_b_fp["r:psfFlux"]) & np.isfinite(df_b_fp["r:psfFluxErr"])

            ax.errorbar(
                dt_src[mask_src],
                df_b_src["r:psfFlux"][mask_src],
                yerr=df_b_src["r:psfFluxErr"][mask_src],
                color=color,
                **style["src"],
            )

            ax.errorbar(
                dt_fp[mask_fp],
                df_b_fp["r:psfFlux"][mask_fp],
                yerr=df_b_fp["r:psfFluxErr"][mask_fp],
                color=color,
                **style["fp"],
            )

            ax.axhline(0, color="black", lw=0.6, ls="--", alpha=0.4)
            ax.set_ylabel("Flux (nJy)")

        elif mode == "mag" and "mag" in df_b.columns:
            mask_src = np.isfinite(df_b_src["mag"]) & np.isfinite(df_b_src["mag_err"])
            mask_fp = np.isfinite(df_b_fp["mag"]) & np.isfinite(df_b_fp["mag_err"])

            ax.errorbar(
                dt_src[mask_src],
                df_b_src["mag"][mask_src],
                yerr=df_b_src["mag_err"][mask_src],
                color=color,
                **style["src"],
            )

            ax.errorbar(
                dt_fp[mask_fp],
                df_b_fp["mag"][mask_fp],
                yerr=df_b_fp["mag_err"][mask_fp],
                color=color,
                **style["fp"],
            )

            ax.invert_yaxis()
            ax.set_ylabel("AB mag")

        elif mode == "luptitude":
            mask_src = np.isfinite(df_b_src["r:psfFlux"]) & np.isfinite(df_b_src["r:psfFluxErr"])
            mask_fp = np.isfinite(df_b_fp["r:psfFlux"]) & np.isfinite(df_b_fp["r:psfFluxErr"])

            if mask_src.sum() >= 3:
                lup_src, lup_err_src, _ = flux_to_luptitude(
                    df_b_src["r:psfFlux"][mask_src],
                    df_b_src["r:psfFluxErr"][mask_src],
                )
                lup_fp, lup_err_fp, _ = flux_to_luptitude(
                    df_b_fp["r:psfFlux"][mask_fp],
                    df_b_fp["r:psfFluxErr"][mask_fp],
                )

                ax.errorbar(dt_src[mask_src], lup_src, yerr=lup_err_src, color=color, **style["src"])
                ax.errorbar(dt_fp[mask_fp], lup_fp, yerr=lup_err_fp, color=color, **style["fp"])

                ax.invert_yaxis()
                ax.set_ylabel("Luptitude")

        ax.set_xlabel("Δt (days)")
        if not is_combined:
            ax.set_title(f"{mode} — band {band}", fontsize=8, color=color)

    # ── Per-band plots ───────────────────────────────────────
    if show_per_band:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = df_obj_src[df_obj_src["r:band"] == band]
            df_b_fp = df_obj_fp[df_obj_fp["r:band"] == band]

            color = band_colors.get(band, "gray")

            for mode in modes:
                ax = axes[current_row][mode_to_col[mode]]
                plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color)

            current_row += 1

    # ── Combined plot ────────────────────────────────────────
    if show_combined:
        for band in bands_obj:
            df_b = df_obj[df_obj["r:band"] == band]
            df_b_src = df_obj_src[df_obj_src["r:band"] == band]
            df_b_fp = df_obj_fp[df_obj_fp["r:band"] == band]

            color = band_colors.get(band, "gray")

            for mode in modes:
                ax = axes[current_row][mode_to_col[mode]]
                plot_one(ax, df_b, df_b_src, df_b_fp, mode, band, color, is_combined=True)

        for mode in modes:
            axes[current_row][mode_to_col[mode]].set_title(f"{mode} (all bands)")
            ax = axes[current_row][mode_to_col[mode]]
            # simple toggle on y axis
            if mode in ("mag", "luptitude"):
                ax.invert_yaxis()
            # really force the inversion
            # if mode in ("mag", "luptitude"):
            #    ax.set_ylim(ax.get_ylim()[::-1])

    # ── Final touches ────────────────────────────────────────
    fig.suptitle(f"diaObjectId = {target_oid} | group = {target_group}", fontsize=11)

    plt.tight_layout()

    if save:
        plt.savefig(f"{filename_prefix}_{target_oid}.png")

    plt.show()

In [ ]:
# ── Choose the tag and object to inspect ──────────────────────────────────────
TARGET_TAG = all_groups[0]  # ← change here
TOP_OIDS = rank_oids(group_oids[TARGET_TAG], nc=5)
TARGET_OID = TOP_OIDS[0]  # ← change here (pick from TOP_OIDS)

print(f"Tag      : {TARGET_TAG}")
print(f"Object   : {TARGET_OID}")
print(f"DDF      : {lc_cache[TARGET_OID]['field']}")
print(f"Top OIDs : {TOP_OIDS}")

In [ ]:
plot_singleobjlightcurve(
    lc_cache,
    TARGET_GROUP,
    TARGET_OID,
    BANDS,
    BAND_COLORS,
    modes=("flux", "mag", "luptitude"),
    # modes=("flux",),
    show_per_band=True,
    show_combined=True,
)

In [ ]:
assert False

In [ ]:
# ── Choose the tag and object to inspect ──────────────────────────────────────
TARGET_TAG = all_groups[0]  # ← change here
TOP_OIDS = rank_oids(group_oids[TARGET_TAG], nc=5)
TARGET_OID = TOP_OIDS[0]  # ← change here (pick from TOP_OIDS)

print(f"Tag      : {TARGET_TAG}")
print(f"Object   : {TARGET_OID}")
print(f"DDF      : {lc_cache[TARGET_OID]['field']}")
print(f"Top OIDs : {TOP_OIDS}")

# ── Build combined light curve ────────────────────────────────────────────────
d = lc_cache[TARGET_OID]
frames = [df for df in (d["fp"], d["src"]) if not df.empty]
df_obj = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    .dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"])
    .sort_values("r:midpointMjdTai")
    .reset_index(drop=True)
)

print(f"\nData points per band:")
print(df_obj.groupby("r:band").size().rename("n_pts").to_frame())

In [ ]:
# Detailed flux + magnitude + luptitude three-panel plot per band
bands_obj = [b for b in BANDS if b in df_obj["r:band"].values]

fig, axes = plt.subplots(len(bands_obj), 3, figsize=(16, 3.2 * len(bands_obj)), squeeze=False)

for row_idx, band in enumerate(bands_obj):
    df_b = df_obj[df_obj["r:band"] == band].copy()
    dt = df_b["r:midpointMjdTai"].values
    dt = dt - dt.min()
    color = BAND_COLORS.get(band, "gray")

    # --- Panel 0: PSF flux (nJy) ---
    ax0 = axes[row_idx][0]
    mask_f = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
    ax0.errorbar(
        dt[mask_f],
        df_b["r:psfFlux"].values[mask_f],
        yerr=df_b["r:psfFluxErr"].values[mask_f],
        fmt="o",
        ms=3,
        lw=0.8,
        elinewidth=0.8,
        color=color,
        alpha=0.85,
    )
    rms = rms_variability(df_b["r:psfFlux"].values[mask_f])
    ax0.axhline(0.0, color="black", lw=0.6, ls="--", alpha=0.4)  # zero-flux reference
    ax0.set_ylabel(f"Flux (nJy) — band {band}", color=color)
    ax0.set_xlabel("Δt (days)")
    ax0.set_title(f"Flux  N={mask_f.sum()}  σ/<f>={rms:.4f}", fontsize=8, color=color)

    # --- Panel 1: AB magnitude (NaN for non-positive flux) ---
    ax1 = axes[row_idx][1]
    if "mag" in df_b.columns and "mag_err" in df_b.columns:
        mask_m = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
        ax1.errorbar(
            dt[mask_m],
            df_b["mag"].values[mask_m],
            yerr=df_b["mag_err"].values[mask_m],
            fmt="o",
            ms=3,
            lw=0.8,
            elinewidth=0.8,
            color=color,
            alpha=0.85,
        )
        ax1.invert_yaxis()  # astronomical convention: brighter = top
        ax1.set_ylabel(f"AB mag — band {band}", color=color)
        ax1.set_xlabel("Δt (days)")
        ax1.set_title(f"AB mag  N={mask_m.sum()} (neg. flux dropped)", fontsize=8, color=color)
    else:
        ax1.set_visible(False)

    # --- Panel 2: Luptitude / asinh magnitude ---
    # All finite measurements are shown, including those with negative flux.
    # This is the recommended representation for DIA differential fluxes.
    ax2 = axes[row_idx][2]
    mask_l = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
    if mask_l.sum() >= 3:
        lup, lup_err, b_soft = flux_to_luptitude(
            df_b["r:psfFlux"].values[mask_l],
            df_b["r:psfFluxErr"].values[mask_l],
        )
        ax2.errorbar(
            dt[mask_l],
            lup,
            yerr=lup_err,
            fmt="o",
            ms=3,
            lw=0.8,
            elinewidth=0.8,
            color=color,
            alpha=0.85,
        )
        ax2.invert_yaxis()  # brighter = top, same convention as magnitudes
        ax2.set_ylabel(f"Luptitude — band {band}", color=color)
        ax2.set_xlabel("Δt (days)")
        ax2.set_title(f"Luptitude  N={mask_l.sum()}  b={b_soft:.1f} nJy", fontsize=8, color=color)
    else:
        ax2.set_visible(False)

field_label = f" | DDF={lc_cache[TARGET_OID]['field']}" if lc_cache[TARGET_OID].get("field") else ""
fig.suptitle(
    f"diaObjectId = {TARGET_OID}  |  tag = {TARGET_TAG}{field_label}",
    fontsize=11,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
savefig(f"03_detail_{TARGET_OID}")
plt.show()

## 11. Summary scatter plot — variability vs mean flux per tag

In [ ]:
if df_flat.empty:
    print("No flatness data available.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"].replace("_", "\n"),
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle(
        "Fink tag variability summary (objects in DDFs)",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("04_tag_summary_scatter")
    plt.show()

    print("\nTop rows sorted by (band, median_rms):")
    display(summary.sort_values(["band", "median_rms"]).head(40))

## 12. Final ranking table — variability by tag

In [ ]:
if not df_ranking.empty:
    print("Ranking by photometric variability (ascending RMS) — all Fink tags in DDFs:")
    print("=" * 110)
    cols = [
        c
        for c in ["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]
        if c in df_ranking.columns
    ]
    print(df_ranking[cols].to_string(index=False, float_format="{:.4f}".format))

    # ── Breakdown by (tag, DDF) ───────────────────────────────────────────────
    if not df_flat.empty and "field" in df_flat.columns:
        print("\nMedian RMS by (tag, DDF):")
        breakdown = (
            df_flat.groupby(["group", "field"])
            .agg(n_obj=("diaObjectId", "nunique"), median_rms=("rms_var", "median"))
            .reset_index()
            .sort_values(["group", "median_rms"])
        )
        display(breakdown)
elif not df_flat.empty:
    # Recompute ranking on the fly from flatness metrics
    print("tag_ranking.csv not found — recomputing from flatness_metrics.csv.")
    ranking = (
        df_flat.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)
    ranking["description"] = ranking["group"].map(FINK_TAGS).fillna("")
    display(ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag", "description"]])
else:
    print("No ranking data available.")